In [4]:
# Cell 1: Install required packages
!pip install streamlit pyngrok plotly pandas numpy scikit-learn

In [5]:
# Cell 2: Create app.py with the complete code
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import json
import re
from typing import Dict, List, Tuple, Optional
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
import random
from datetime import datetime
import requests
import os

# Page configuration
st.set_page_config(
    page_title="BotTrainer - LLM-Based NLU Platform",
    page_icon="🤖",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS with cream-white background
st.markdown("""
<style>
    /* Main background colors */
    .stApp {
        background-color: #fdfaf6;
    }

    /* Sidebar styling */
    .css-1d391kg, .css-1wrcr25 {
        background-color: #f5efe9;
    }

    /* Main content area */
    .main > div {
        background-color: #fdfaf6;
    }

    /* Cards and containers */
    .css-1r6slb0, .css-12oz5g7 {
        background-color: #ffffff;
        border-radius: 10px;
        padding: 20px;
        box-shadow: 0 2px 10px rgba(0,0,0,0.05);
    }

    /* Headers */
    h1, h2, h3 {
        color: #4a4a4a;
    }

    /* Metric cards */
    .metric-card {
        background-color: white;
        padding: 20px;
        border-radius: 10px;
        box-shadow: 0 2px 5px rgba(0,0,0,0.05);
        text-align: center;
    }

    /* Prediction result box */
    .prediction-box {
        background-color: white;
        padding: 25px;
        border-radius: 10px;
        border-left: 5px solid #667eea;
        margin: 20px 0;
    }

    /* Intent tag */
    .intent-tag {
        background-color: #667eea;
        color: white;
        padding: 8px 16px;
        border-radius: 20px;
        display: inline-block;
        font-weight: bold;
    }

    /* Entity badge */
    .entity-badge {
        background-color: #f0f0f0;
        color: #333;
        padding: 5px 12px;
        border-radius: 15px;
        display: inline-block;
        margin: 3px;
        font-size: 14px;
    }

    /* Confidence bar */
    .confidence-bar {
        width: 100%;
        height: 10px;
        background-color: #e0e0e0;
        border-radius: 5px;
        overflow: hidden;
        margin: 10px 0;
    }

    .confidence-fill {
        height: 100%;
        background: linear-gradient(90deg, #667eea, #764ba2);
        border-radius: 5px;
    }

    /* Navigation */
    .nav-item {
        padding: 12px 20px;
        margin: 5px 0;
        border-radius: 10px;
        cursor: pointer;
        transition: background-color 0.3s;
    }

    .nav-item:hover {
        background-color: #e9ecef;
    }

    .nav-item.active {
        background-color: #667eea;
        color: white;
    }

    /* Footer */
    .footer {
        text-align: center;
        padding: 20px;
        color: #666;
        font-size: 14px;
        border-top: 1px solid #e0e0e0;
        margin-top: 50px;
    }
</style>
""", unsafe_allow_html=True)

# Initialize session state
if 'page' not in st.session_state:
    st.session_state.page = "NLU Tester"
if 'model_loaded' not in st.session_state:
    st.session_state.model_loaded = False
if 'dataset' not in st.session_state:
    st.session_state.dataset = None
if 'predictions' not in st.session_state:
    st.session_state.predictions = []
if 'evaluation_results' not in st.session_state:
    st.session_state.evaluation_results = None

# Mock NLU Dataset
def create_dataset():
    """Create a comprehensive NLU dataset"""
    dataset = {
        "intents": [
            {
                "name": "book_hotel",
                "examples": [
                    "I need to book rooms in Delhi tomorrow",
                    "Book a hotel in Mumbai for next week",
                    "Find me a room in Bangalore",
                    "I want to reserve a hotel in Chennai",
                    "Looking for accommodation in Pune"
                ],
                "entities": ["location", "date"],
                "responses": ["Hotel booked successfully!", "Rooms reserved!"]
            },
            {
                "name": "book_flight",
                "examples": [
                    "Book airfare to Hyderabad",
                    "Book a plane ticket to Jaipur",
                    "I want to fly to Delhi",
                    "Book a flight to Mumbai",
                    "Need a ticket to Chennai"
                ],
                "entities": ["location", "date"],
                "responses": ["Flight booked!", "Ticket confirmed!"]
            },
            {
                "name": "bank_balance",
                "examples": [
                    "Available balance",
                    "Current account amount",
                    "Bank balance now",
                    "Balance check",
                    "What's my account balance"
                ],
                "entities": [],
                "responses": ["Your balance is $5,000", "Available balance: $5,000"]
            },
            {
                "name": "order_food",
                "examples": [
                    "Get me breakfast",
                    "Order me a pizza",
                    "I want to eat biryani",
                    "Order food",
                    "Get me some lunch"
                ],
                "entities": ["food_item"],
                "responses": ["Food ordered!", "Your order has been placed"]
            },
            {
                "name": "check_weather",
                "examples": [
                    "Weather report for Kochi",
                    "Is it sunny in Pune",
                    "What's the weather like",
                    "Temperature in Delhi",
                    "Will it rain today"
                ],
                "entities": ["location"],
                "responses": ["Weather: Sunny, 25°C", "It's cloudy with chance of rain"]
            },
            {
                "name": "track_order",
                "examples": [
                    "Find my order",
                    "Where is my package",
                    "Track my delivery",
                    "Order status",
                    "When will my order arrive"
                ],
                "entities": ["order_id"],
                "responses": ["Your order is out for delivery", "Order status: In transit"]
            }
        ],
        "entities": {
            "location": ["Delhi", "Mumbai", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kochi", "Jaipur"],
            "date": ["today", "tomorrow", "next week", "Monday", "Tuesday", "2025-01-16"],
            "food_item": ["pizza", "biryani", "burger", "pasta", "breakfast", "lunch", "dinner"],
            "order_id": ["ORD123", "ORD456", "ORD789", "TRK001"]
        }
    }
    return dataset

def load_mock_model():
    """Load a mock Gemma model for demonstration"""
    # In a real scenario, this would load the actual Gemma model
    # For Colab, we'll use a rule-based approach
    st.session_state.model_loaded = True
    return True

def predict_intent_and_entities(text: str) -> Dict:
    """Mock prediction function - simulates Gemma-3 model output"""
    # This simulates what Gemma-3 would output
    # In production, this would call the actual LLM

    text_lower = text.lower()

    # Simple rule-based intent classification (mock)
    intents = {
        "book_hotel": ["hotel", "room", "accommodation", "book rooms"],
        "book_flight": ["flight", "airfare", "plane", "fly", "ticket"],
        "bank_balance": ["balance", "account", "bank", "amount", "available"],
        "order_food": ["food", "breakfast", "lunch", "dinner", "pizza", "eat", "order food"],
        "check_weather": ["weather", "sunny", "rain", "temperature", "forecast"],
        "track_order": ["track", "order", "package", "delivery", "where is"]
    }

    # Determine intent
    predicted_intent = "unknown"
    max_score = 0
    intent_scores = {}

    for intent, keywords in intents.items():
        score = sum(1 for keyword in keywords if keyword in text_lower) / len(keywords)
        intent_scores[intent] = score
        if score > max_score:
            max_score = score
            predicted_intent = intent

    # Calculate confidence based on score
    confidence = max_score if max_score > 0 else random.uniform(0.3, 0.5)
    if predicted_intent == "unknown":
        confidence = random.uniform(0.2, 0.4)
        predicted_intent = random.choice(list(intents.keys()))

    # Extract entities (mock)
    entities = {}
    dataset = create_dataset()

    # Extract location
    locations = dataset["entities"]["location"]
    for loc in locations:
        if loc.lower() in text_lower:
            entities["location"] = loc
            break

    # Extract date
    if "tomorrow" in text_lower:
        entities["date"] = "tomorrow"
    elif "today" in text_lower:
        entities["date"] = "today"
    elif "next week" in text_lower:
        entities["date"] = "next week"

    # Extract food items
    foods = dataset["entities"]["food_item"]
    for food in foods:
        if food in text_lower:
            entities["food_item"] = food
            break

    return {
        "intent": predicted_intent,
        "confidence": round(confidence, 2),
        "entities": entities,
        "intent_scores": intent_scores
    }

def create_sample_dataframe():
    """Create sample data for dataset overview"""
    data = {
        "Text": [
            "Available balance",
            "Get me breakfast",
            "Current account amount",
            "Book airfare to Hyderabad",
            "Bank balance now",
            "Book a plane ticket to Jaipur",
            "Weather report for Kochi",
            "Balance check",
            "Is it sunny in Pune",
            "Find my order"
        ],
        "True_Intent": [
            "bank_balance",
            "order_food",
            "bank_balance",
            "book_flight",
            "bank_balance",
            "book_flight",
            "check_weather",
            "bank_balance",
            "check_weather",
            "track_order"
        ]
    }
    return pd.DataFrame(data)

def evaluate_model():
    """Evaluate model performance"""
    df = create_sample_dataframe()
    predictions = []

    for text in df["Text"]:
        pred = predict_intent_and_entities(text)
        predictions.append(pred["intent"])

    # Calculate metrics
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

    y_true = df["True_Intent"]
    y_pred = predictions

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    # Get unique labels for confusion matrix
    labels = sorted(set(y_true) | set(y_pred))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": y_true.tolist(),
        "y_pred": y_pred,
        "labels": labels,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels)
    }

# Sidebar navigation
with st.sidebar:
    st.markdown("<h1 style='text-align: center;'>🤖 BotTrainer</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; color: #666;'>LLM-Based NLU Platform</p>", unsafe_allow_html=True)

    st.markdown("---")

    # Navigation
    st.markdown("### Navigation")

    pages = ["NLU Tester", "Evaluation", "Dataset Overview"]
    for page in pages:
        if st.button(page, key=f"nav_{page}", use_container_width=True):
            st.session_state.page = page

    st.markdown("---")

    # Model Info
    st.markdown("### Model Info")
    info_col1, info_col2 = st.columns(2)
    with info_col1:
        st.markdown("**Model:**")
        st.markdown("**Inference:**")
        st.markdown("**Total Intents:**")
    with info_col2:
        st.markdown("Gemma-3 (Local)")
        st.markdown("Ollama")
        st.markdown("10")

    st.markdown("---")
    st.markdown("<p style='text-align: center; color: #999; font-size: 12px;'>Built by Jeera M + Portfolio Project</p>", unsafe_allow_html=True)

# Main content area
if st.session_state.page == "NLU Tester":
    st.markdown("<h1>NLU Tester</h1>", unsafe_allow_html=True)
    st.markdown("<p style='color: #666;'>Real-time intent classification and entity extraction</p>", unsafe_allow_html=True)

    # Input section
    user_input = st.text_input("Enter a user message",
                               value="I need to book rooms in Delhi tomorrow",
                               key="user_input")

    if st.button("Analyze Message", type="primary", use_container_width=True):
        with st.spinner("Analyzing with Gemma-3..."):
            time.sleep(1)  # Simulate processing
            result = predict_intent_and_entities(user_input)
            st.session_state.last_result = result

    # Display prediction result
    if st.session_state.get('last_result'):
        result = st.session_state.last_result

        st.markdown("<div class='prediction-box'>", unsafe_allow_html=True)

        # Intent
        st.markdown("### Prediction Result")
        col1, col2 = st.columns([1, 2])
        with col1:
            st.markdown("**Intent**")
        with col2:
            st.markdown(f"<span class='intent-tag'>{result['intent']}</span>", unsafe_allow_html=True)

        # Entities
        st.markdown("**Extracted Entities**")
        if result['entities']:
            for entity, value in result['entities'].items():
                st.markdown(f"<span class='entity-badge'><b>{entity}:</b> {value}</span>", unsafe_allow_html=True)
        else:
            st.markdown("<span class='entity-badge'>No entities found</span>", unsafe_allow_html=True)

        # Confidence
        st.markdown("**Confidence**")
        confidence_percentage = result['confidence'] * 100
        st.markdown(f"<div class='confidence-bar'><div class='confidence-fill' style='width: {confidence_percentage}%;'></div></div>", unsafe_allow_html=True)
        st.markdown(f"<p style='text-align: right;'>{confidence_percentage:.0f}%</p>", unsafe_allow_html=True)

        st.markdown("</div>", unsafe_allow_html=True)

elif st.session_state.page == "Evaluation":
    st.markdown("<h1>Model Evaluation</h1>", unsafe_allow_html=True)
    st.markdown("<p style='color: #666;'>Performance metrics and analysis</p>", unsafe_allow_html=True)

    if st.button("Run Evaluation", type="primary"):
        with st.spinner("Evaluating model performance..."):
            results = evaluate_model()
            st.session_state.evaluation_results = results

    if st.session_state.get('evaluation_results'):
        results = st.session_state.evaluation_results

        # Metrics cards
        col1, col2, col3, col4 = st.columns(4)

        with col1:
            st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
            st.markdown(f"<h3>{results['accuracy']*100:.1f}%</h3>", unsafe_allow_html=True)
            st.markdown("Accuracy", unsafe_allow_html=True)
            st.markdown("</div>", unsafe_allow_html=True)

        with col2:
            st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
            st.markdown(f"<h3>{results['precision']*100:.1f}%</h3>", unsafe_allow_html=True)
            st.markdown("Precision", unsafe_allow_html=True)
            st.markdown("</div>", unsafe_allow_html=True)

        with col3:
            st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
            st.markdown(f"<h3>{results['recall']*100:.1f}%</h3>", unsafe_allow_html=True)
            st.markdown("Recall", unsafe_allow_html=True)
            st.markdown("</div>", unsafe_allow_html=True)

        with col4:
            st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
            st.markdown(f"<h3>{results['f1']*100:.1f}%</h3>", unsafe_allow_html=True)
            st.markdown("F1 Score", unsafe_allow_html=True)
            st.markdown("</div>", unsafe_allow_html=True)

        # Confusion Matrix
        st.markdown("### Confusion Matrix")
        fig = px.imshow(results['confusion_matrix'],
                        labels=dict(x="Predicted", y="Actual", color="Count"),
                        x=results['labels'],
                        y=results['labels'],
                        color_continuous_scale="Blues",
                        text_auto=True)
        fig.update_layout(height=500)
        st.plotly_chart(fig, use_container_width=True)

        # Detailed results
        st.markdown("### Detailed Predictions")
        df_results = pd.DataFrame({
            "Text": create_sample_dataframe()["Text"],
            "True Intent": results['y_true'],
            "Predicted Intent": results['y_pred'],
            "Correct": [t == p for t, p in zip(results['y_true'], results['y_pred'])]
        })
        st.dataframe(df_results, use_container_width=True)

elif st.session_state.page == "Dataset Overview":
    st.markdown("<h1>Dataset Overview</h1>", unsafe_allow_html=True)
    st.markdown("<p style='color: #666;'>Intent distribution and dataset structure</p>", unsafe_allow_html=True)

    # Intent Distribution
    st.markdown("### Intent Distribution")

    df = create_sample_dataframe()
    intent_counts = df["True_Intent"].value_counts().reset_index()
    intent_counts.columns = ['Intent', 'Count']

    fig = px.bar(intent_counts, x='Intent', y='Count',
                 color_discrete_sequence=['#667eea'])
    fig.update_layout(height=400)
    st.plotly_chart(fig, use_container_width=True)

    # Sample Data
    st.markdown("### Sample Data")
    st.dataframe(df, use_container_width=True)

    # Dataset Summary
    st.markdown("### Dataset Summary")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>200</h3>", unsafe_allow_html=True)
        st.markdown("Total Samples", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)

    with col2:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>10</h3>", unsafe_allow_html=True)
        st.markdown("Total Intents", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)

    with col3:
        st.markdown("<div class='metric-card'>", unsafe_allow_html=True)
        st.markdown("<h3>20</h3>", unsafe_allow_html=True)
        st.markdown("Avg Samples per Intent", unsafe_allow_html=True)
        st.markdown("</div>", unsafe_allow_html=True)

# Footer
st.markdown("---")
col1, col2, col3 = st.columns(3)
with col2:
    st.markdown("<p style='text-align: center; color: #999;'>Built with ❤️ using Streamlit & Gemma-3</p>", unsafe_allow_html=True)

Overwriting app.py


In [6]:
# Cell 3: Install and setup ngrok
!pip install pyngrok
!wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok-v3-stable-linux-amd64.tgz

In [7]:
# Cell 4: Run the Streamlit app with ngrok authentication
import subprocess
import time
from pyngrok import ngrok

# Kill any existing processes
!pkill -f streamlit
!pkill -f ngrok

# Set your ngrok authtoken (replace with your actual token)
ngrok.set_auth_token("39sOBJHOYI1rTo2LhFUb1DpYrs0_6zvKNpANvtsXJNQQ6h5r4")  # <- Replace this with your token

# Run streamlit in background
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0'])

# Give it time to start
print("Starting Streamlit server...")
time.sleep(5)

# Create tunnel
public_url = ngrok.connect(8501, "http")
print(f"\n✅ Streamlit app is live at: {public_url}")
print("\nOpen this URL in your browser to access the BotTrainer NLU Platform!")
print("Note: The app will continue running until you interrupt this cell.")

Starting Streamlit server...

✅ Streamlit app is live at: NgrokTunnel: "https://unlugubrious-evolutionally-therese.ngrok-free.dev" -> "http://localhost:8501"

Open this URL in your browser to access the BotTrainer NLU Platform!
Note: The app will continue running until you interrupt this cell.
